# Workshop 05 — Logistic Regression & Classification

**Machine Learning for Media Production | Guided practical (~90 minutes)**

In Workshop 04, you used **linear regression** to predict a number. Today, you will use **logistic regression** to predict one of two classes.

**Learning goals:**
1. Recognise `X` (features), `y` (labels), and their array shapes.
2. Explain how `sigmoid` converts a linear score into a value between 0 and 1.
3. Calculate **binary cross-entropy (log loss)** for a prediction.
4. Train a classifier and understand its **decision boundary**.
5. Change the **classification threshold** and inspect classification errors.

**Materials:** The class PPT explains the theory. This Notebook is the hands-on companion, not a second lecture.

**How to use:** Run cells **in order**. Cells marked **YOUR TURN** contain intentional `NotImplementedError` placeholders; replace those lines before continuing. Short written questions should be answered in the Markdown cells below them.

**Environment:** Choose the Python environment you used for Workshop 04. If a library is missing, activate the project's `.venv` and run this in a terminal (not as a Python cell):

```bash
uv pip install numpy matplotlib scikit-learn ipykernel
```

No `sudo` is needed. No downloaded dataset or external API is required.

## Section 1 — From Regression to Classification (10 min)

Linear regression predicts a **continuous value**: $\hat y=wx+b$.

Logistic regression still computes a linear score, but then transforms it into a value between 0 and 1:

$$z=w_1x_1+w_2x_2+b, \qquad p=\sigma(z)$$

Here `p` represents the model's estimated probability of **class 1**. With a threshold of 0.5, we predict class 1 when $p\geq0.5$; otherwise we predict class 0.

We will use **synthetic data** with two numerical features (`feature 1`, `feature 2`). The labels are just `0` and `1`; they do **not** represent real medical or media data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

np.set_printoptions(precision=3, suppress=True)

X, y = make_classification(
    n_samples=150,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=0.85,
    flip_y=0.08,
    random_state=42,
)

print('X shape:', X.shape)
print('y shape:', y.shape)
print('First row of X:', X[0])
print('First label:', y[0])
print('Class counts [0, 1]:', np.bincount(y))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.7))
scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', vmin=0, vmax=1,
                     s=38, edgecolor='black', linewidth=0.35, alpha=0.85)
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.set_title('Synthetic data: two classes')
ax.grid(alpha=0.15)
plt.show()

**CHECK 1 — Write your answer (1–2 sentences).**

- What do `150` and `2` mean in `X.shape == (150, 2)`?
- Why is `y.shape == (150,)`, rather than `(150, 2)`?

**My answer:** ...

## Section 2 — Sigmoid and Probability (15 min)

The model starts with a linear score $z$. Sigmoid transforms it:

$$\sigma(z)=\frac{1}{1+e^{-z}}$$

- Large **negative** $z$ → probability near `0`.
- $z=0$ → probability `0.5`.
- Large **positive** $z$ → probability near `1`.

You do not need to derive the formula today. You **do** need to connect the formula to NumPy code.

### YOUR TURN 1 — Complete `sigmoid(z)`

Use `np.exp(...)`. Your function should work for both a **single number** and a **NumPy array**. Replace the placeholder, then run the test cell.

In [ ]:
def sigmoid(z):
    # TODO: Return 1 / (1 + np.exp(-z))
    raise NotImplementedError('Complete sigmoid(z) before continuing')

In [ ]:
# Check your function (do not change this cell)
assert np.isclose(sigmoid(0), 0.5)
assert np.allclose(sigmoid(np.array([-2.0, 0.0, 2.0])),
                   [0.11920292, 0.5, 0.88079708], atol=1e-6)
print('Sigmoid checks passed!')
for value in [-2, 0, 2]:
    print(f'z={value:+d} -> p={float(sigmoid(value)):.3f}')

In [ ]:
z_values = np.linspace(-6, 6, 300)
p_values = sigmoid(z_values)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(z_values, p_values, label='sigmoid(z)', linewidth=2)
ax.axhline(0.5, color='grey', linestyle='--', label='threshold = 0.5')
ax.axvline(0, color='grey', linestyle=':', linewidth=1)
ax.set(xlabel='Linear score z', ylabel='Probability of class 1',
       title='Sigmoid turns a score into a probability', ylim=(-0.05, 1.05))
ax.legend()
ax.grid(alpha=0.15)
plt.show()

**CHECK 2 — Think before you run:** Is `sigmoid(-2)` above or below 0.5? Does `sigmoid(2)` predict class 0 or 1 with a threshold of 0.5?

**My answer:** ...

## Section 3 — Binary Cross-Entropy / Log Loss (20 min)

A model needs a way to measure **how wrong its predicted probability is**. For one binary label $y\in\{0,1\}$:

$$L=-[y\ln(p)+(1-y)\ln(1-p)]$$

For many samples, we take the **mean** of these losses. In NumPy, `np.log` means natural logarithm $\ln$.

Notice the two cases:

- When the true label is $y=1$, loss is $-\ln(p)$.
- When the true label is $y=0$, loss is $-\ln(1-p)$.

Very confident **wrong** predictions receive a large loss. We use `np.clip` so that a probability of exactly 0 or 1 does not create `log(0)`.

### YOUR TURN 2 — Complete `binary_cross_entropy(y_true, p)`

The safety step is already provided. Complete only the `return` expression using `np.mean` and `np.log`.

**Hint:** `y_true` and `p` are arrays of the same length.

In [ ]:
def binary_cross_entropy(y_true, p):
    y_true = np.asarray(y_true, dtype=float)
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-12, 1 - 1e-12)
    # TODO: Return the mean of -[y*log(p) + (1-y)*log(1-p)]
    raise NotImplementedError('Complete binary_cross_entropy(...)')

In [ ]:
# Check your loss function (do not change)
assert np.isclose(binary_cross_entropy([1], [0.9]), -np.log(0.9))
assert np.isclose(binary_cross_entropy([0], [0.9]), -np.log(0.1))
assert np.isclose(binary_cross_entropy([1, 0], [0.9, 0.2]),
                  (-np.log(0.9) - np.log(0.8)) / 2)
print('Log loss checks passed!')

examples = [(1, 0.9), (1, 0.1), (0, 0.9), (0, 0.1)]
for true_label, probability in examples:
    loss = binary_cross_entropy([true_label], [probability])
    print(f'y={true_label}, p={probability:.1f} -> loss={loss:.3f}')

In [ ]:
p_grid = np.linspace(0.01, 0.99, 200)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(p_grid, -np.log(p_grid), label='True label y = 1')
ax.plot(p_grid, -np.log(1-p_grid), label='True label y = 0')
ax.set(xlabel='Predicted probability p of class 1', ylabel='Loss',
       title='Binary cross-entropy penalises confident mistakes')
ax.legend()
ax.grid(alpha=0.15)
plt.show()

**CHECK 3 — Write your answer:** For a sample whose true label is `1`, which prediction is worse: `p=0.9` or `p=0.1`? Explain using the **loss values** above.

**My answer:** ...

## Section 4 — Train a Logistic Regression Model (20 min)

The PPT introduces gradient descent. It works by repeatedly adjusting the weights and bias to reduce loss, just as in Workshop 04.

The key difference is the prediction and loss:

| | Workshop 04: Linear Regression | Workshop 05: Logistic Regression |
|---|---|---|
| Prediction | $\hat y=wx+b$ | $p=\sigma(w^Tx+b)$ |
| Typical loss | MSE | Binary Cross-Entropy |
| Update idea | reduce loss by changing weights | same idea |

**Today:** Use `scikit-learn` to train the model. We are not implementing the full optimiser from scratch.

We split the data into a **training set** (to fit the model) and a **test set** (to check it on unseen samples). Do not train on the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=7, stratify=y
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# sklearn returns probabilities for class 0 and class 1.
# [:, 1] means: probability of class 1, for every test sample.
test_probabilities = model.predict_proba(X_test)[:, 1]
test_predictions = model.predict(X_test)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Weights:', model.coef_[0])
print('Bias:', model.intercept_[0])
print('First 5 probabilities of class 1:', np.round(test_probabilities[:5], 3))
print('First 5 predicted labels:', test_predictions[:5])
print('Test accuracy:', round(accuracy_score(y_test, test_predictions), 3))

# The default sklearn prediction uses a 0.5 threshold for this binary classifier.
assert np.array_equal(test_predictions, (test_probabilities >= 0.5).astype(int))

### See the decision boundary

The dark line shows where the model predicts `p = 0.5`. On one side of the line the model predicts class 0, and on the other side it predicts class 1.

In [ ]:
x1_range = np.linspace(X[:, 0].min()-1, X[:, 0].max()+1, 200)
x2_range = np.linspace(X[:, 1].min()-1, X[:, 1].max()+1, 200)
x1_grid, x2_grid = np.meshgrid(x1_range, x2_range)
grid_points = np.column_stack([x1_grid.ravel(), x2_grid.ravel()])
grid_prob = model.predict_proba(grid_points)[:, 1].reshape(x1_grid.shape)

fig, ax = plt.subplots(figsize=(6.8, 5))
ax.contourf(x1_grid, x2_grid, grid_prob, levels=[0, 0.5, 1],
            colors=['#cfe8fa', '#fde0db'], alpha=0.7)
ax.contour(x1_grid, x2_grid, grid_prob, levels=[0.5], colors='black', linewidths=2)
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm',
           vmin=0, vmax=1, edgecolor='black', linewidth=0.3, s=27)
ax.set(xlabel='Feature 1', ylabel='Feature 2', title='Learned linear decision boundary (p = 0.5)')
plt.show()

**CHECK 4 — Write your answer:**

- How many values are in `model.coef_[0]`? Why?
- Is the decision boundary a straight line or a curve? (We used two original features, with no polynomial features.)

**My answer:** ...

## Section 5 — Thresholds and Evaluation (20 min)

The model outputs a probability. A **threshold** decides how we convert that probability into a class.

$$\hat y=\begin{cases}1 & \text{if }p\geq t\\0 & \text{if }p<t\end{cases}$$

The default is often $t=0.5$, but other thresholds may be useful. A higher threshold makes it **harder to predict class 1**. It does not automatically make the model more accurate.

**Important terms:**
- **False Positive (FP):** predicted 1, actual 0.
- **False Negative (FN):** predicted 0, actual 1.

For a confusion matrix produced by sklearn, use rows = **actual**, columns = **predicted**, in class order 0 then 1.

### YOUR TURN 3 — Complete `predict_with_threshold(p, threshold)`

Use a NumPy comparison (`>=`) and convert the Boolean values to integers with `.astype(int)`.

In [ ]:
def predict_with_threshold(p, threshold):
    p = np.asarray(p)
    # TODO: Return an array of 0/1 labels using your threshold.
    raise NotImplementedError('Complete predict_with_threshold(...)')

In [ ]:
# Check your threshold function
assert np.array_equal(predict_with_threshold([0.2, 0.5, 0.8], 0.5), [0, 1, 1])
assert np.array_equal(predict_with_threshold([0.2, 0.5, 0.8], 0.7), [0, 0, 1])
print('Threshold checks passed!')

for threshold in [0.3, 0.5, 0.7]:
    predicted = predict_with_threshold(test_probabilities, threshold)
    tn, fp, fn, tp = confusion_matrix(y_test, predicted, labels=[0, 1]).ravel()
    print(f't={threshold:.1f} | accuracy={accuracy_score(y_test, predicted):.3f} | '
          f'predicted class 1={predicted.sum():2d} | FP={fp:2d} | FN={fn:2d}')

**CHECK 5 — Write your answer:**

Compare `threshold = 0.5` and `threshold = 0.7`.

1. Does the number of predicted class-1 samples increase or decrease?
2. What happened to **FP** and **FN** in **this dataset**?
3. Is a higher threshold always better? Why not?

**My answer:** ...

### Inspect actual mistakes

The following cell finds samples that were classified incorrectly at threshold 0.5. Each row refers to the **test set**, not the original full dataset.

In [ ]:
wrong_indices = np.flatnonzero(test_predictions != y_test)
print('Number of mistakes:', len(wrong_indices))

for i in wrong_indices[:5]:
    print(f'test row {i:2d} | features={np.round(X_test[i], 2)} | '
          f'actual={y_test[i]} | p(class 1)={test_probabilities[i]:.3f} | '
          f'predicted={test_predictions[i]}')

**CHECK 6 — Answer in your own words:** Select one mistake above. Report its test row, true label, probability of class 1, and predicted label. Is the output a probability or an absolute guarantee?

**My answer:** ...

## Wrap-up (5 min)

You now have the whole workflow:

$$\text{features }X\;\longrightarrow\;z=w^TX+b\;\longrightarrow\;\sigma(z)=p\;\longrightarrow\;\text{class}\;0\text{ or }1$$

During training, the model adjusts its weights to reduce **binary cross-entropy**. During prediction, we can change the **classification threshold** without retraining the model.

### Before leaving

- [ ] I can explain `X.shape` and `y.shape`.
- [ ] My three `YOUR TURN` functions pass their checks.
- [ ] I can explain a decision boundary and a classification threshold.
- [ ] I can distinguish FP from FN.

**Next:** Assignment 2 uses a **different synthetic dataset**. Use this Notebook as a reference, but submit your own results and short explanations.

**Optional challenge (not required):** Inspect `model.coef_` and `model.intercept_`. Calculate $z$ and $\sigma(z)$ for one test row manually with NumPy, then compare with `model.predict_proba(...)`.